# 4.4: What pretrained vectors show

Real embeddings trained on a large corpus show the frequency effect directly. The code below loads the 100,000 most frequent words of a set of pretrained GloVe vectors (Pennington et al. 2014), 50-dimensional vectors trained on Wikipedia and newswire text, and lists the nearest neighbors (Section 4.3) of two ambiguous words:


In [ ]:
import gzip
import numpy as np

# GloVe vectors (50 dimensions, trained on Wikipedia and newswire text), top 100,000 words
words, vecs = [], []
with gzip.open("glove50.gz", "rt", encoding="utf8") as f:
    next(f)                                             # header line: "400000 50"
    for i, line in enumerate(f):
        if i == 100_000:
            break
        parts = line.rstrip().split(" ")
        words.append(parts[0])
        vecs.append(np.array(parts[1:], dtype=np.float32))
E = np.stack(vecs)
E /= np.linalg.norm(E, axis=1, keepdims=True)          # unit length: dot product = cosine
idx = {w: i for i, w in enumerate(words)}

def neighbors(word, k=10):
    scores = E @ E[idx[word]]
    order = [i for i in np.argsort(-scores) if words[i] != word][:k]
    return [words[i] for i in order]

def cos(a, b):
    return float(E[idx[a]] @ E[idx[b]])

for w in ["bank", "apple"]:
    print(f"{w}: {', '.join(neighbors(w, 8))}")
for a, b in [("bank", "loan"), ("bank", "river"), ("apple", "microsoft"), ("apple", "pear")]:
    print(f"cos({a}, {b}) = {cos(a, b):.3f}")


**Expected output**

Output:

```text
bank: banks, securities, banking, investment, exchange, financial, credit, lender
apple: blackberry, chips, iphone, microsoft, ipad, pc, ipod, intel
cos(bank, loan) = 0.676
cos(bank, river) = 0.414
cos(apple, microsoft) = 0.733
cos(apple, pear) = 0.543
```
